# NovaMarket Perú — Notebook Reproducible CRISP-DM

Este notebook ejecuta y audita de punta a punta las **6 fases de CRISP-DM** sobre transacciones reales (`data/raw/online_retail.csv`):
1. Comprensión del negocio y linaje SHA-256
2. Análisis exploratorio de datos (EDA)
3. Limpieza, Reglas de Asociación (Market Basket) y Features RFM con corte temporal (`2011-09-09`)
4. Entrenamiento K-Means + 3 clasificadores supervisados
5. Evaluación en holdout y explicabilidad SHAP
6. Verificación de inferencia compatible con `POST /predict` y `POST /api/recommend`

In [ ]:
import json
from pathlib import Path
import joblib
import pandas as pd

ROOT = Path("..").resolve()
eda = json.loads((ROOT / "reports" / "eda_summary.json").read_text(encoding="utf-8"))
metrics = json.loads((ROOT / "reports" / "metrics_summary.json").read_text(encoding="utf-8"))
rules_df = pd.read_csv(ROOT / "reports" / "top_association_rules.csv")

print("=== RESUMEN EDA ===")
print(json.dumps(eda, indent=2, ensure_ascii=False))
print("\n=== MODELO GANADOR Y METRICAS ===")
print(json.dumps(metrics["cv_comparison"], indent=2, ensure_ascii=False))
rules_df.head(8)

In [ ]:
# Prueba de inferencia directa sobre artefactos serializados
churn_model = joblib.load(ROOT / "models" / "modelo_recompra_clientes.joblib")
rfm_bundle = joblib.load(ROOT / "models" / "modelo_segmentacion_rfm.joblib")
basket_rules = json.loads((ROOT / "models" / "basket_rules.json").read_text(encoding="utf-8"))

sample = pd.DataFrame([{
    "recency_days": 12,
    "frequency": 18,
    "monetary": 6450.0,
    "avg_basket_value": 358.33,
    "total_items": 1200,
    "unique_products": 95,
    "tenure_days": 240,
    "purchase_velocity": 2.0,
    "cancellation_count": 0,
    "is_uk": "Internacional"
}])
prob = float(churn_model.predict_proba(sample)[0, 1])
print(f"Probabilidad de recompra (90d): {prob:.4f}")
print("Top 3 recomendaciones para SKU 22698:")
for r in basket_rules.get("22698", [])[:3]:
    print(f"  -> {r['consequent_name']} | Conf={r['confidence']:.2%} | Lift={r['lift']:.2f}x")